# CPTEC-WRF for Joinville — download & process **hourly rainfall + 2-m temperature + 10-m wind**

**What this notebook does.** Downloads the operational **CPTEC/INPE WRF (AMS 7 km)** forecast files
for a chosen run, extracts them over the **Joinville grid**, and writes a clean, self-describing
**NetCDF** with the fields the dashboard and the verification work need, on an hourly time axis:

- **`precip_mm_h`** — hourly precipitation (mm h⁻¹), correctly *de-accumulated*;
- **`t2m_degC`** — 2-metre air temperature (°C), *instantaneous*;
- **`wspd10_ms` / `wdir10_deg`** (+ components `u10_ms`, `v10_ms`) — 10-metre wind speed and
  meteorological *from*-direction, *instantaneous*.

Consolidates the Rio downloader + the earlier rain+temperature Joinville notebook, and **adds wind**.
Feeds both the *Previsão* dashboard page (per-basin forecast) and the Stage-A verification
(`Joinville_LAB_LOG`).

---

### The one scientific idea: *accumulated* vs *instantaneous* fields

NWP models write some fields as **running accumulations from init** and others as **instantaneous
snapshots**. They must be processed **differently** — mixing them silently corrupts the data.

| Field | Stored as | To get the hourly value you… | Why |
|---|---|---|---|
| **Precipitation** (`tp`, `acpcp`, `ncpcp`) | **accumulated since init** (mm, ever-growing) | **difference** successive hours: `rain(H) = tp(H) − tp(H−1)` | totals only increase; the *rate* is the increment |
| **2-m temperature** (`t2m`) | **instantaneous** (K snapshot) | take **as-is**, convert K → °C | already the value *at* that valid time |
| **10-m wind** (`u10`, `v10`) | **instantaneous** (m s⁻¹ snapshot) | take **as-is**; speed `√(u²+v²)`, dir `(270−atan2(v,u))` | already *at* that time; components combine into speed/dir — **never differenced** |

The notebook detects the precipitation accumulation convention empirically and de-accumulates it,
while leaving temperature and wind untouched apart from unit/derivation. (WRF Users' Guide; ECMWF
accumulated-variables docs; cfgrib `stepType`/`stepRange`.)

> ⚠️ **Retention caveat.** CPTEC keeps only a limited history of raw runs; old dates may return
> HTTP 404/403 — the download cell prints per-file status. Pick a **recent** event (within retention),
> or drop archived GRIB2 into `/content/joinville_wrf/`.

## 1. Install the libraries

Installs `cfgrib`/`eccodes`/`ecmwflibs` (read WRF GRIB2), `xarray`, `requests`, `netcdf4`. If Colab
shows **RESTART RUNTIME**, click it, then continue from section 2. Run once per session.

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=False)
pip('cfgrib','eccodes','ecmwflibs','xarray','netcdf4','requests')
try:
    import cfgrib  # noqa
    print('\u2705 cfgrib import OK')
except Exception as e:
    print('cfgrib import failed, trying system eccodes:', e)
    subprocess.run(['apt-get','install','-y','-q','libeccodes0'], check=False)
    import cfgrib  # noqa
    print('\u2705 cfgrib import OK (after apt eccodes)')

## 2. Configuration — run & Joinville grid (**EDIT these**)

CPTEC initialises at **00Z and 12Z** only, and keeps just a short history of raw runs. So instead of
guessing a run hour, leave **`AUTO_LATEST = True`** and the notebook will **probe the server and pick
the most recent run that is actually available** (section 3) — the run date/hour adjust themselves.
Set `AUTO_LATEST = False` only for a **specific case-study event** (recent, within retention) or when
you have archived GRIB2 for a past date, and then fill `RUN_DATE`/`RUN_HOUR`.

`FORECAST_HOURS` are the lead times (h from init) to fetch — a consecutive range. Because rainfall
de-accumulation differences successive hours, **the first listed lead is the baseline**, so hourly
rainfall starts at the *second* lead (temperature/wind exist at every lead).

**Domain.** Default is the established Joinville verification box (lat ∈ [−28.61, −24.00],
lon ∈ [−51.41, −46.28]; `Joinville_LAB_LOG` §E2.3). The dashboard processor
(`build_wrf_basins.py`) crops it to the municipality/basins window, so one file serves both uses.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta

# --- run selection ---
AUTO_LATEST = True                 # True: auto-pick the most recent run AVAILABLE on CPTEC (best for a first/live run).
                                   # False: use the exact RUN_DATE/RUN_HOUR below (specific case-study event / archived files).
RUN_DATE    = '2023-12-30'         # used only if AUTO_LATEST=False  (>>> your Joinville event date, UTC)
RUN_HOUR    = 12                   # used only if AUTO_LATEST=False  (0 or 12 UTC)
FORECAST_HOURS = list(range(0, 25))  # +0…+24 h leads (próximas 24 h; passo de 1 h). O primeiro é a linha-base de acúmulo.

# --- Joinville domain (established verification box; the dashboard processor crops it) ---
LAT_MIN, LAT_MAX = -28.61, -24.00
LON_MIN, LON_MAX = -51.41, -46.28
# --- lighter dashboard-only crop (uncomment to shrink the NetCDF): ---
# LAT_MIN, LAT_MAX = -26.85, -25.80
# LON_MIN, LON_MAX = -49.45, -48.40

BASE_URL   = 'https://dataserver.cptec.inpe.br/dataserver_modelos/wrf/ams_07km/brutos'
OUTPUT_DIR = Path('/content/joinville_wrf'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_dt = datetime.strptime(f'{RUN_DATE} {RUN_HOUR:02d}', '%Y-%m-%d %H')   # manual / fallback run
print(f'AUTO_LATEST   : {AUTO_LATEST}   (if True, the actual run is discovered in section 3)')
print(f'Manual run    : {run_dt:%Y-%m-%d %H}Z   (used only if AUTO_LATEST=False)')
print(f'Lead times    : +{FORECAST_HOURS[0]}\u2026+{FORECAST_HOURS[-1]} h  ({len(FORECAST_HOURS)} steps)')
print(f'Joinville box : lat[{LAT_MIN}, {LAT_MAX}]  lon[{LON_MIN}, {LON_MAX}]')
print(f'Note          : hourly rainfall starts at the 2nd lead (+{FORECAST_HOURS[1]} h); temp/wind exist at every lead.')

## 3. Find the latest available run (if `AUTO_LATEST`) + download the GRIB2 files

If `AUTO_LATEST=True`, this cell first **probes CPTEC for the most recent run that actually exists** —
walking back over the 00Z/12Z runs and checking the file for the last requested lead with a tiny
ranged request — and sets `run_dt` to it, so you never guess a run hour. Then it downloads each lead,
**skipping files already present**, printing the **HTTP status and size for every file**. If nothing
downloads, the run is outside retention or access is restricted.

In [ ]:
import requests
HEADERS = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/120 Safari/537.36'}

def build_url(run_dt, fcast_dt):
    r = run_dt.strftime('%Y%m%d%H'); f = fcast_dt.strftime('%Y%m%d%H')
    y,m,d,h = run_dt.strftime('%Y'),run_dt.strftime('%m'),run_dt.strftime('%d'),run_dt.strftime('%H')
    return f'{BASE_URL}/{y}/{m}/{d}/{h}/WRF_cpt_07KM_{r}_{f}.grib2'

sess = requests.Session(); sess.headers.update(HEADERS)

# --- auto-detect the most recent AVAILABLE run (probe the last requested lead with a ranged GET) ---
def _run_available(rd, probe_fh):
    url = build_url(rd, rd + timedelta(hours=probe_fh))
    try:
        r = sess.get(url, stream=True, timeout=40, headers={'Range': 'bytes=0-1'})
        ok = r.status_code in (200, 206); r.close(); return ok
    except Exception:
        return False

def find_latest_run(fhs, max_back_hours=72):
    now = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
    anchor = now.replace(hour=(12 if now.hour >= 12 else 0))      # most recent 00/12Z <= now
    for k in range(max_back_hours // 12 + 1):
        cand = anchor - timedelta(hours=12 * k)
        if _run_available(cand, fhs[-1]):
            return cand
    return None

if AUTO_LATEST:
    print('Auto-detecting the latest available CPTEC run\u2026')
    latest = find_latest_run(FORECAST_HOURS, max_back_hours=72)
    if latest is not None:
        run_dt = latest
        print(f'   >>> latest available run: {run_dt:%Y-%m-%d %H}Z')
    else:
        print(f'   \u26a0\ufe0f none found in the last 72 h \u2014 using manual {run_dt:%Y-%m-%d %H}Z '
              '(set AUTO_LATEST=False or check CPTEC status/retention).')
print(f'Run           : {run_dt:%Y-%m-%d %H}Z')
print(f'Valid times   : {run_dt+timedelta(hours=FORECAST_HOURS[0]):%Y-%m-%d %H}Z \u2026 '
      f'{run_dt+timedelta(hours=FORECAST_HOURS[-1]):%Y-%m-%d %H}Z\n')

downloaded, status = [], []
for fh in FORECAST_HOURS:
    fdt = run_dt + timedelta(hours=fh)
    url = build_url(run_dt, fdt)
    dest = OUTPUT_DIR / Path(url).name
    if dest.exists() and dest.stat().st_size > 1e5:
        downloaded.append((fh, dest)); status.append((fh,'cached',dest.stat().st_size)); continue
    try:
        r = sess.get(url, stream=True, timeout=180)
        if r.status_code == 200:
            with open(dest,'wb') as f:
                for ch in r.iter_content(1<<16): f.write(ch)
            sz = dest.stat().st_size
            if sz > 1e5: downloaded.append((fh,dest)); status.append((fh,'200 OK',sz))
            else: status.append((fh,'200 but tiny',sz)); dest.unlink(missing_ok=True)
        else:
            status.append((fh,f'HTTP {r.status_code}',0))
    except Exception as e:
        status.append((fh,f'ERR {e}',0))

print('FH   status         size')
for fh,st,sz in status:
    print(f'+{fh:<3d} {st:<14s} {sz/1e6:6.1f} MB' if sz else f'+{fh:<3d} {st}')
print(f'\n\u2705 usable files: {len(downloaded)}')
if not downloaded:
    print('\n\u26a0\ufe0f  Nothing downloaded. Likely outside CPTEC retention, or access restricted. '
          'Try RUN_HOUR=0, a more recent date, or supply archived GRIB2 manually.')

## 4. Extract rainfall + temperature + wind over the Joinville box + read GRIB metadata

Opens every GRIB2 file and pulls the **precipitation** fields (`tp`, `acpcp`, `ncpcp`), the **2-m
temperature** (`t2m`/`2t`/`t@2 m`), and the **10-m wind components** (`u10`/`10u`, `v10`/`10v`), then
crops to the Joinville box. GRIB2 splits variables across messages by level/type, so
`cfgrib.open_datasets` returns several sub-datasets; this cell searches them, drops level/time scalar
coords so the fields merge cleanly on lat/lon, and assembles one dataset per lead time. It also prints
`tp`'s `stepType`/`stepRange` (used next to de-accumulate). Warnings print if temp or wind are absent
(rainfall still proceeds).

In [ ]:
import cfgrib, numpy as np, xarray as xr

RAIN_VARS = ['tp','acpcp','ncpcp']
WIND = {'u10': ['u10','10u'], 'v10': ['v10','10v']}

def _norm(ds):
    '''normalise longitudes to -180..180, sort, crop to the Joinville box.'''
    lon = ds.longitude
    if float(lon.max()) > 180:
        ds = ds.assign_coords(longitude=(((lon+180)%360)-180)).sortby('longitude')
    ds = ds.sortby('latitude')
    return ds.sel(latitude=slice(LAT_MIN,LAT_MAX), longitude=slice(LON_MIN,LON_MAX))

def _clean(da):
    '''keep only latitude/longitude dims; drop scalar level/time coords so vars merge cleanly.'''
    return da.reset_coords(drop=True)

def open_joinville(path):
    '''Dataset with tp/acpcp/ncpcp/t2m/u10/v10 on lat/lon, cropped to Joinville, + tp GRIB attrs.'''
    dss = cfgrib.open_datasets(str(path))
    rain = temp = None; wind = {}; tp_attrs = None
    for d in dss:
        if rain is None and any(v in d.data_vars for v in RAIN_VARS):
            rain = _norm(d[[v for v in RAIN_VARS if v in d.data_vars]])
        if temp is None:
            if 't2m' in d.data_vars:      temp = _norm(d[['t2m']])
            elif '2t' in d.data_vars:     temp = _norm(d[['2t']].rename({'2t':'t2m'}))
            elif 't' in d.data_vars and 'heightAboveGround' in d['t'].coords \
                 and float(np.atleast_1d(d['t'].heightAboveGround.values)[0]) == 2.0:
                temp = _norm(d[['t']].rename({'t':'t2m'}))
        for canon, names in WIND.items():
            if canon in wind: continue
            for nm in names:
                if nm in d.data_vars:
                    wind[canon] = _norm(d[[nm]].rename({nm:canon})); break
    if rain is None:
        return None, None
    if tp_attrs is None and 'tp' in rain:
        tp_attrs = {k:v for k,v in rain['tp'].attrs.items() if 'step' in k.lower() or 'Grib' in k}
    parts = [_clean(rain[v]).rename(v) for v in rain.data_vars]
    if temp is not None:
        parts.append(_clean(temp['t2m']).reindex_like(rain, method='nearest').rename('t2m'))
    else:
        print('   \u26a0\ufe0f t2m not found in this file — temperature missing for this step')
    for canon in ('u10','v10'):
        if canon in wind:
            parts.append(_clean(wind[canon][canon]).reindex_like(rain, method='nearest').rename(canon))
    if 'u10' not in wind or 'v10' not in wind:
        print('   \u26a0\ufe0f 10-m wind (u10/v10) not both found — wind missing for this step')
    return xr.merge(parts), tp_attrs

per_fh = []; grib_attrs = None; had_temp = True; had_wind = True
for fh, path in downloaded:
    ds, tp_attrs = open_joinville(path)
    if ds is None:
        print(f'  \u26a0\ufe0f +{fh}h: no precip vars found'); continue
    if grib_attrs is None and tp_attrs: grib_attrs = tp_attrs
    if 't2m' not in ds: had_temp = False
    if not ('u10' in ds and 'v10' in ds): had_wind = False
    per_fh.append(ds.expand_dims(forecast_hour=[fh]))

assert per_fh, 'No datasets extracted — check section 3 downloaded real GRIB2 files.'
# keep only variables present in every step, so concat aligns
common = set(per_fh[0].data_vars)
for ds in per_fh[1:]: common &= set(ds.data_vars)
per_fh = [ds[list(common)] for ds in per_fh]
wrf = xr.concat(per_fh, dim='forecast_hour').sortby('forecast_hour')
wrf = wrf.assign_coords(valid_time=('forecast_hour',
      [run_dt+timedelta(hours=int(h)) for h in wrf.forecast_hour.values]))
print('Joinville subset grid :', dict(wrf.sizes))
print('Variables present     :', list(wrf.data_vars))
print('t2m in all / wind in all:', had_temp, '/', had_wind)
print('GRIB step metadata (tp):', grib_attrs)

## 5. Process — de-accumulate rain (→ mm h⁻¹); temperature & wind as-is (instantaneous)

**Rainfall (accumulated → hourly):** check `tp ≈ acpcp + ncpcp`; detect the accumulation convention
empirically (domain-mean monotonic + `stepRange` starting at `0`); if accumulated-from-init,
difference successive hours (consuming the first lead) and clip rounding negatives.

**Temperature (instantaneous → °C):** no differencing — convert K → °C only (detected by magnitude),
with a physical-range sanity flag.

**Wind (instantaneous):** no differencing — keep the components `u10`, `v10` (m s⁻¹) and derive
**speed** `√(u²+v²)` and the meteorological **from-direction** `(270 − atan2(v,u)) mod 360`.

In [ ]:
import numpy as np

# ---- (a) rainfall: internal consistency tp ≈ acpcp + ncpcp ----
if all(v in wrf for v in ['tp','acpcp','ncpcp']):
    resid = float(np.abs(wrf['tp'] - (wrf['acpcp']+wrf['ncpcp'])).max())
    print(f'[rain] max |tp-(acpcp+ncpcp)| = {resid:.4f} mm  (\u22480 expected)')
else:
    print('[rain] not all of tp/acpcp/ncpcp present; skipping sum check')

# ---- (b) rainfall: determine accumulation convention ----
dom_mean = wrf['tp'].mean(('latitude','longitude')).values
monotonic = bool(np.all(np.diff(dom_mean) >= -1e-6))
step_range = (grib_attrs or {}).get('GRIB_stepRange','?')
from_init = monotonic or (isinstance(step_range,str) and step_range.strip().startswith('0'))
print(f'[rain] domain-mean tp per step: {np.round(dom_mean,3)}')
print(f'[rain] monotonic? {monotonic} | stepRange={step_range}')
print(f'   >>> convention: {"ACCUMULATED FROM INIT (will difference)" if from_init else "PER-STEP (use tp directly)"}')

# ---- (c) rainfall: de-accumulate to hourly mm/h ----
if from_init:
    tp_hourly = wrf['tp'].diff('forecast_hour')       # aligned to later step; drops the first lead
    neg = int((tp_hourly < -1e-6).sum()); tp_hourly = tp_hourly.clip(min=0)
    print(f'[rain] differenced; negatives clipped: {neg} cells (\u22480 if truly from-init)')
    wrf_h = wrf.isel(forecast_hour=slice(1,None)).copy()
    wrf_h['precip_mm_h'] = tp_hourly
else:
    wrf_h = wrf.copy(); wrf_h['precip_mm_h'] = wrf['tp']
wrf_h['precip_mm_h'].attrs.update(units='mm h-1', long_name='WRF hourly precipitation (Joinville)',
    convention=('accumulated_from_init->differenced' if from_init else 'per_step'))

# ---- (d) temperature: instantaneous, K -> C (NO differencing) ----
if 't2m' in wrf_h:
    tK = wrf_h['t2m']; is_kelvin = float(np.nanmedian(tK.values)) > 100.0
    t2m_c = (tK - 273.15) if is_kelvin else tK
    wrf_h['t2m_degC'] = t2m_c
    wrf_h['t2m_degC'].attrs.update(units='degC', long_name='WRF 2-m air temperature (Joinville)',
        field_type='instantaneous', note=('converted from Kelvin' if is_kelvin else 'already Celsius'))
    lo, hi = float(np.nanmin(t2m_c)), float(np.nanmax(t2m_c))
    flag = '' if (-10 <= lo and hi <= 50) else '  \u26a0\ufe0f outside plausible surface range!'
    print(f'[temp] instantaneous 2-m T: {lo:.1f}\u2026{hi:.1f} \u00b0C (kelvin_input={is_kelvin}){flag}')
else:
    print('[temp] no t2m — temperature output skipped')

# ---- (e) wind: instantaneous components (m/s); speed + from-direction (NO differencing) ----
if 'u10' in wrf_h and 'v10' in wrf_h:
    u, v = wrf_h['u10'], wrf_h['v10']
    wrf_h['u10_ms'] = u; wrf_h['v10_ms'] = v
    wrf_h['wspd10_ms'] = np.hypot(u, v)
    wrf_h['wdir10_deg'] = (270.0 - np.degrees(np.arctan2(v, u))) % 360.0   # meteorological FROM-direction
    for nm, ln, un in [('u10_ms','10-m U wind','m s-1'),('v10_ms','10-m V wind','m s-1'),
                       ('wspd10_ms','10-m wind speed','m s-1'),('wdir10_deg','10-m wind direction (from)','deg')]:
        wrf_h[nm].attrs.update(units=un, long_name='WRF '+ln+' (Joinville)', field_type='instantaneous')
    print(f'[wind] instantaneous 10-m wind: speed {float(wrf_h.wspd10_ms.min()):.1f}\u2026{float(wrf_h.wspd10_ms.max()):.1f} m/s')
else:
    print('[wind] no u10/v10 — wind output skipped')

print('\n\u2705 hourly product valid times:', [str(v)[:16] for v in wrf_h.valid_time.values])
print('   output variables:', [v for v in ['precip_mm_h','t2m_degC','wspd10_ms','wdir10_deg','u10_ms','v10_ms'] if v in wrf_h])

## 6. Save the clean NetCDF

Writes hourly **rainfall (mm h⁻¹)**, instantaneous **2-m temperature (°C)** and **10-m wind**
(speed, direction, and u/v components), one field per valid hour, to `wrf_joinville_<run>Z.nc` — data
stored together with lat/lon/valid-time coordinates and units.

**Upload this file** to the dashboard chat (or repo `site/data/`); `build_wrf_basins.py` turns it into
the per-basin forecast + the *Previsão* map. It is also the WRF input for Stage-A verification.

In [ ]:
out = OUTPUT_DIR / f'wrf_joinville_{run_dt:%Y%m%d%H}Z.nc'
keep_vars = [v for v in ['precip_mm_h','t2m_degC','wspd10_ms','wdir10_deg','u10_ms','v10_ms'] if v in wrf_h]
keep = wrf_h[keep_vars].copy()
keep.attrs.update(source='CPTEC/INPE WRF AMS 7km', run_time=str(run_dt),
                  domain=f'Joinville (lat {LAT_MIN}..{LAT_MAX}, lon {LON_MIN}..{LON_MAX})',
                  note='precip_mm_h=de-accumulated hourly; t2m_degC & wind=instantaneous')
keep.to_netcdf(out)
print(f'\u2705 saved {out}  ({out.stat().st_size/1e6:.2f} MB)')
for v in keep_vars:
    print(f'   {v:12s} dims={dict(keep[v].sizes)}  units={keep[v].attrs.get("units","?")}')
print('\n>>> Upload this .nc to the dashboard chat — build_wrf_basins.py does the rest.')

## 7. Sanity look (rainfall + temperature + wind)

Six quick panels — a five-second check before trusting the numbers: rainfall (series + map),
temperature (series + map), and wind (speed series + speed map with direction arrows). No mapping
libraries needed; a PNG is saved.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np

vt = [str(v)[5:16] for v in wrf_h.valid_time.values]; tgt = len(vt)//2
have_t = 't2m_degC' in wrf_h; have_w = 'wspd10_ms' in wrf_h
fig, ax = plt.subplots(3, 2, figsize=(13, 13))

p = wrf_h['precip_mm_h']
ax[0,0].plot(vt, p.mean(('latitude','longitude')).values, 'o-', label='box mean')
ax[0,0].plot(vt, p.max(('latitude','longitude')).values, 's--', label='box max')
ax[0,0].set_title('(a) hourly precip'); ax[0,0].set_ylabel('mm h$^{-1}$'); ax[0,0].legend(); ax[0,0].tick_params(axis='x', rotation=45)
im = ax[0,1].pcolormesh(p.longitude, p.latitude, p.isel(forecast_hour=tgt).values,
                        vmin=0, vmax=float(np.nanpercentile(p.values,99)) or 1, cmap='viridis')
ax[0,1].set_title(f'(b) precip @ {vt[tgt]}'); plt.colorbar(im, ax=ax[0,1])

if have_t:
    t = wrf_h['t2m_degC']
    ax[1,0].plot(vt, t.mean(('latitude','longitude')).values, 'o-', color='#c0392b', label='box mean')
    ax[1,0].fill_between(range(len(vt)), t.min(('latitude','longitude')).values, t.max(('latitude','longitude')).values,
                         color='#c0392b', alpha=0.15, label='min-max')
    ax[1,0].set_title('(c) 2-m temperature'); ax[1,0].set_ylabel('\u00b0C'); ax[1,0].legend(); ax[1,0].tick_params(axis='x', rotation=45)
    im2 = ax[1,1].pcolormesh(t.longitude, t.latitude, t.isel(forecast_hour=tgt).values, cmap='RdYlBu_r')
    ax[1,1].set_title(f'(d) 2-m T @ {vt[tgt]}'); plt.colorbar(im2, ax=ax[1,1])
else:
    for a in (ax[1,0], ax[1,1]): a.text(.5,.5,'no t2m', ha='center'); a.axis('off')

if have_w:
    w = wrf_h['wspd10_ms']
    ax[2,0].plot(vt, w.mean(('latitude','longitude')).values, 'o-', color='#2c7fb8', label='box mean')
    ax[2,0].plot(vt, w.max(('latitude','longitude')).values, 's--', color='#2c7fb8', label='box max')
    ax[2,0].set_title('(e) 10-m wind speed'); ax[2,0].set_ylabel('m s$^{-1}$'); ax[2,0].legend(); ax[2,0].tick_params(axis='x', rotation=45)
    sp = w.isel(forecast_hour=tgt).values
    im3 = ax[2,1].pcolormesh(w.longitude, w.latitude, sp, cmap='YlGnBu', vmin=0)
    U = wrf_h['u10_ms'].isel(forecast_hour=tgt).values; V = wrf_h['v10_ms'].isel(forecast_hour=tgt).values
    s = max(1, sp.shape[0]//12)
    ax[2,1].quiver(w.longitude.values[::s], w.latitude.values[::s], U[::s,::s], V[::s,::s], scale=200, width=0.003)
    ax[2,1].set_title(f'(f) 10-m wind @ {vt[tgt]}'); plt.colorbar(im3, ax=ax[2,1])
else:
    for a in (ax[2,0], ax[2,1]): a.text(.5,.5,'no wind', ha='center'); a.axis('off')

plt.tight_layout(); plt.savefig(str(OUTPUT_DIR/'wrf_joinville_sanity.png'), dpi=110, bbox_inches='tight')
plt.show(); print('\u2705 saved sanity figure ->', OUTPUT_DIR/'wrf_joinville_sanity.png')

## Notes, caveats & next step

- **Accumulated vs instantaneous — the recurring trap.** Many clipped negatives while inferring
  *from-init* is contradictory: inspect `grib_attrs`/raw `tp`. Temperature and wind are instantaneous
  — **never difference them**.
- **`tmax`/`tmin`, gusts.** Some products carry interval max/min temperature or wind gusts — those are
  *interval statistics*, not instantaneous; handle separately if needed. This notebook uses
  instantaneous `t2m` and mean 10-m wind.
- **Lead-time asymmetry.** CPTEC runs 00/12Z only, so an evening valid time is a several-hour lead —
  a short-range forecast, not a nowcast. State this in any verification.
- **Next step.** Upload `wrf_joinville_<run>Z.nc`. The dashboard's `build_wrf_basins.py` area-weights
  it onto the 7 basins (rain total, mean temperature, mean wind) and builds the *Previsão* map with the
  municipality outline; the same file feeds Stage-A verification (`Joinville_LAB_LOG`).